# Inspect AMASS sequences

Browse the raw AMASS (SMPL-X) sequences from `amass_raw` (see `config/paths.yaml`), compress them (crop the start and the end, keep every n-th frame) and play the original next to the compressed version.

- **Top:** skeleton, original (gray, left) and compressed (blue, right), world frame with z up. Press **Play** or drag the frame slider. The skeleton is the mean body from `smpl_skeleton.npz`, not the subject's real shape.
- **Bottom:** joint speed (mean over the 21 body joints, deg/s) and root speed (m/s) over time, original vs compressed. Red = cropped, dashed line = idle threshold, green = first/last frame above it, black = current frame.
- The compressed skeleton holds each kept frame until the next one, so playback shows what the model would see.

Needs `pip install -e ".[dev]"` (adds `ipywidgets`).

In [ ]:
import ipywidgets as w

from hp.data import explorer as ex

parents, p3d0 = ex.load_skeleton()
view = ex.SkeletonView(parents, p3d0)
scan_cache, seq_cache = {}, {}
state = {"seq": None, "comp": None, "series": None, "title": "", "busy": False}


def scan(dataset):
    if dataset not in scan_cache:
        scan_cache[dataset] = ex.scan_dataset(dataset)
    return scan_cache[dataset]


def load(entry):
    if entry["path"] not in seq_cache:
        if len(seq_cache) > 20:
            seq_cache.pop(next(iter(seq_cache)))
        seq_cache[entry["path"]] = ex.load_sequence(entry["path"])
    return seq_cache[entry["path"]]

In [ ]:
wide, mid = w.Layout(width="620px"), w.Layout(width="420px")
dataset = w.Dropdown(options=ex.list_datasets(), description="Dataset", layout=mid)
subject = w.Dropdown(description="Subject", layout=mid)
sequence = w.Dropdown(description="Sequence", layout=wide)
frame = w.IntSlider(description="Frame", min=0, max=1, continuous_update=True, layout=wide)
play = w.Play(interval=60, min=0, max=1, step=1)
w.jslink((play, "value"), (frame, "value"))
speed = w.Dropdown(options=[0.25, 0.5, 1, 2], value=1, description="Speed", layout=w.Layout(width="150px"))
stride = w.IntSlider(value=1, min=1, max=60, description="Every n-th", continuous_update=False, layout=mid)
crop_start = w.FloatSlider(value=0, min=0, max=1, step=0.05, description="Crop start [s]", continuous_update=False, layout=wide)
crop_end = w.FloatSlider(value=0, min=0, max=1, step=0.05, description="Crop end [s]", continuous_update=False, layout=wide)
thr = w.FloatSlider(value=10.0, min=1, max=200, step=1, description="Idle thr", continuous_update=False, layout=mid)
apply_btn = w.Button(description="Apply suggested crop")
skeleton_img = w.Image(format="png", layout=w.Layout(width="1000px"))
series_img = w.Image(format="png", layout=w.Layout(width="1000px"))


def update_frame(change=None):
    if state["seq"] is None or state["busy"]:
        return
    skeleton_img.value = view.png(state["seq"], state["comp"], frame.value)
    series_img.value = state["series"].png(frame.value)


def refresh(change=None):
    if state["seq"] is None or state["busy"]:
        return
    state["comp"] = ex.compress(state["seq"], stride.value, crop_start.value, crop_end.value)
    state["series"] = ex.SeriesView(state["seq"], state["comp"], thr.value, title=state["title"])
    update_frame()


def set_speed(change=None):
    if state["seq"] is not None:
        play.step = max(1, round(state["seq"]["fps"] * play.interval / 1000 * speed.value))


def on_dataset(change=None):
    subs = sorted({e["subject"] for e in scan(dataset.value)})
    subject.options = subs
    subject.value = subs[0]
    on_subject()


def on_subject(change=None):
    entries = [e for e in scan(dataset.value) if e["subject"] == subject.value]
    sequence.options = [
        (f"{e['name'].replace('_stageii.npz', '')}  ({e['n_frames']} fr, {e['n_frames'] / e['fps']:.1f} s)", e)
        for e in entries
    ]
    sequence.value = sequence.options[0][1]


def on_sequence(change=None):
    e = sequence.value
    if e is None:
        return
    seq = load(e)
    n, dur = len(seq["poses"]), len(seq["poses"]) / seq["fps"]
    keep_start, keep_end = crop_start.value, crop_end.value  # crop and stride are kept across sequences
    state["busy"] = True
    state["seq"] = seq
    state["title"] = f"{dataset.value}/{e['subject']}/{e['name'].replace('_stageii.npz', '')[:40]}"
    play.max = frame.max = max(n - 1, 1)
    crop_start.max = crop_end.max = round(max(dur, 0.05), 2)
    frame.value = 0
    crop_start.value, crop_end.value = keep_start, keep_end  # clipped if the sequence is shorter
    state["busy"] = False
    set_speed()
    refresh()


def on_apply(_):
    seq = state["seq"]
    n, fps = len(seq["poses"]), seq["fps"]
    sug = ex.suggest_crop(ex.motion_energy(seq["poses"], fps), fps, thr.value)
    state["busy"] = True
    if sug is None:
        crop_start.value = crop_end.value = 0
    else:
        crop_start.value, crop_end.value = round(sug[0] / fps, 2), round((n - 1 - sug[1]) / fps, 2)
    state["busy"] = False
    refresh()


dataset.observe(on_dataset, "value")
subject.observe(on_subject, "value")
sequence.observe(on_sequence, "value")
for widget in (stride, crop_start, crop_end, thr):
    widget.observe(refresh, "value")
frame.observe(update_frame, "value")
speed.observe(set_speed, "value")
apply_btn.on_click(on_apply)
on_dataset()

w.VBox(layout=w.Layout(align_items="flex-start"), children=[
    dataset, subject, sequence,
    w.HBox([frame, play, speed]),
    stride, crop_start, crop_end,
    w.HBox([thr, apply_btn]),
    skeleton_img, series_img,
])

## How much is lost?

Compares the original frames with the compressed ones (the current *Every n-th* and crop settings) for the selected sequence or for **all sequences of the selected subject**. Poses are the 19 joint rotations HuProSO models.

- **Recall (nearest neighbour):** for a sample of original poses, the geodesic distance to the nearest kept pose, summed over the 19 joints as in HuProSO and shown per joint in degrees. Gray = same number of randomly chosen frames, for comparison.
- **Per-joint Wasserstein (W1):** distance between the pose distributions of original and compressed. The gray bars are a noise floor (two independent samples of the original). A strided set can fall below it: it is more evenly spread than an independent sample.

With *Auto-crop* each sequence is cropped by its own idle start and end (threshold above) instead of the fixed *Crop start/end*.

In [ ]:
scope = w.ToggleButtons(options=["Sequence", "Subject"], value="Subject", description="Scope")
auto_crop = w.Checkbox(value=False, description="Auto-crop each sequence by the idle threshold", indent=False, layout=wide)
n_ref = w.IntSlider(value=2000, min=500, max=10000, step=500, description="Recall refs", continuous_update=False, layout=mid)
n_w = w.IntSlider(value=1000, min=200, max=3000, step=200, description="W1 samples", continuous_update=False, layout=mid)
run_btn = w.Button(description="Compute", button_style="primary")
report_html = w.HTML()
report_img = w.Image(format="png", layout=w.Layout(width="1000px"))


def on_compute(_):
    if state["seq"] is None:
        return
    report_html.value = "Computing..."
    if scope.value == "Sequence":
        seqs, label = [state["seq"]], state["title"]
    else:
        entries = [e for e in scan(dataset.value) if e["subject"] == subject.value]
        seqs, label = [load(e) for e in entries], f"{dataset.value}/{subject.value} ({len(entries)} sequences)"
    auto = thr.value if auto_crop.value else None
    orig, comp = ex.gather(seqs, stride.value, crop_start.value, crop_end.value, auto_thr_deg_s=auto)
    rep = ex.compression_report(orig, comp, n_ref.value, n_w.value)
    report_html.value = f"<h4>{label}</h4>" + ex.report_table(rep)
    report_img.value = ex.plot_report(rep)


run_btn.on_click(on_compute)
w.VBox(layout=w.Layout(align_items="flex-start"), children=[w.HBox([scope, run_btn]), auto_crop, n_ref, n_w, report_html, report_img])